# Performance Evaluation
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Run alpha attribution against CAPM, FF3, and FF5**
2. **Tell when "alpha" is just exposure to a known factor**
3. **Detect overfitting** in claimed alphas by comparing in-sample to OOS
4. **Apply the Sharpe-ratio standard error formula** to know what's distinguishable from noise
5. **Audit AI-generated alpha claims** — multi-factor benchmarks, sample period choice, multiple testing

## 📋 TOC
1. [Setup](#setup)  2. [The Alpha-Attribution Workflow](#workflow)
3. [Pitfall Checklist](#pitfalls)  4. [Case: Cathie Wood / ARKK](#arkk)
5. [Case: Buffett / Berkshire](#brk)  6. [Sharpe SE and "Is this real?"](#se)
7. [🎯 Challenge: Grading a Fund](#challenge)
8. [Submission](#submit)  9. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## The Alpha-Attribution Workflow <a id="workflow"></a>

You have a fund's return series. You want to answer: *does it have alpha?*

The standard test:

$$r_t^{\text{fund}} = \alpha + \beta_1 \cdot MKT_t + \beta_2 \cdot SMB_t + \beta_3 \cdot HML_t + \beta_4 \cdot RMW_t + \beta_5 \cdot CMA_t + \beta_6 \cdot MOM_t + \epsilon_t$$

- **CAPM** (1 factor): MKT only
- **FF3** (3 factors): MKT, SMB, HML
- **FF5** (5 factors): adds RMW (profitability), CMA (investment)
- **FF6** (6 factors): adds MOM

**The alpha you find shrinks as you add factors.** A fund that has α=8% against
CAPM might have α=2% against FF6. That difference is *factor exposure being
re-classified* as "not alpha."

> **💡 Key principle**
>
> You should benchmark a strategy against factors **it could be plausibly
> exposed to**. Don't benchmark a stock-picker against the bond market;
> don't benchmark a quant equity strategy against the S&P 500 if it has
> small-cap exposure.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Single-factor benchmark** | Find "alpha" that's actually size or value loading | Always run FF3 or FF6 in parallel |
| 2 | **Sample-period cherry-picking** | A fund's alpha was 10% in 2010-2015, -3% in 2016-2020 | Always report stability across sub-samples |
| 3 | **Survivorship in the fund universe** | Studying surviving funds inflates the average alpha by 1-2% | Use vintage data or include discontinued funds |
| 4 | **Multiple testing** | Out of 500 funds, ~25 will show alpha with p<0.05 by chance | Apply Bonferroni or use the Harvey-Liu adjustments |
| 5 | **Reading R² wrong** | High R² doesn't mean no alpha — it means the factors capture most variance | Look at α and its t-stat, not just R² |
| 6 | **t-stat = significance ≠ economic** | t=3 on a 0.1%/yr alpha is meaningless economically | Always look at magnitude alongside t-stat |

---
## Case: Cathie Wood / ARKK <a id="arkk"></a>

ARKK had a wild ride: dominated 2020 (+150% return), then collapsed 2021-2022.
Alpha attribution typically shows:
- Huge positive **MKT** loading (it's leveraged tech)
- Huge positive **SMB** loading (small-cap tilt)
- Negative **HML** loading (anti-value, pro-growth)
- Negative **CMA** loading (high-investment / capex stocks)

After controlling for these, alpha against FF5 is approximately **zero or
negative** depending on the sample window. The "outperformance" was almost
entirely factor exposure.

---
## Case: Buffett / Berkshire <a id="brk"></a>

Frazzini, Kabiller, and Pedersen (2018) decomposed Berkshire's returns:
- Positive **MKT** loading (~0.9)
- Strong **HML** loading (value-tilted)
- Strong **RMW** loading (high-profitability tilt)
- Modest **BAB** loading (Betting-Against-Beta: prefers low-beta stocks)

After controlling for these factors, Berkshire's alpha is **positive but
much smaller** than the raw return suggests. The bulk of Berkshire's
outperformance is **factor exposure that Buffett identified before
academics named the factors.**

---
## Sharpe SE and "Is this real?" <a id="se"></a>

Given a sample of $T$ observations with realized Sharpe $\widehat{SR}$, the
standard error (approximate, large-sample) is:

$$\text{SE}(\widehat{SR}) \approx \sqrt{\frac{1 + \widehat{SR}^2 / 2}{T}}$$

So with 10 years of monthly data (T=120) and SR=1.0:

$$\text{SE} \approx \sqrt{1.5 / 120} \approx 0.11$$

A 1.0 Sharpe estimated over 10 years has SE ≈ 0.11 — so the 95% CI is
roughly [0.78, 1.22]. Plenty of room to be wrong about the magnitude.

In [ ]:
# Sharpe SE calculator
def sharpe_se(sharpe, T):
    return np.sqrt((1 + sharpe**2 / 2) / T)

for sharpe, years, freq in [(1.0, 10, 12), (0.5, 10, 12), (2.0, 5, 252), (0.3, 30, 12)]:
    T = years * freq
    se = sharpe_se(sharpe, T)
    print(f"Sharpe={sharpe}, T={T} ({years}y of {freq}/yr): SE = {se:.3f}, 95% CI = [{sharpe-1.96*se:.2f}, {sharpe+1.96*se:.2f}]")

---
## 🎯 Challenge: Grading a Fund <a id="challenge"></a>

> **Setup.** A fund manager pitches you their track record:
> - 5 years of monthly returns
> - Mean return: 18% / year
> - Volatility: 22% / year
> - CAPM alpha: 4.2%/yr (t-stat 1.8)
> - FF3 alpha: 1.1%/yr (t-stat 0.5)

### Q1 — Sharpe and its standard error

Assume a risk-free rate of 4%/year.

> **📌 Required:**
> ```python
> fund_sharpe        = ____   # (mean - rf) / vol
> fund_sharpe_se     = ____   # use the formula above with T = 60 months
> ```

In [ ]:
mean_ret = 0.18
vol      = 0.22
rf       = 0.04
T = 5 * 12

fund_sharpe = ____
fund_sharpe_se = ____
print(f"Sharpe: {fund_sharpe:.2f} ± {fund_sharpe_se:.2f}")

### Q2 — Alpha shrinkage

Compute the difference between CAPM and FF3 alpha (in percentage points). This is the amount of "alpha" that was actually small/value loading.

> **📌 Required:**
> ```python
> capm_alpha  = 0.042
> ff3_alpha   = 0.011
> alpha_shrinkage = ____   # capm - ff3
> ```

In [ ]:
capm_alpha = 0.042
ff3_alpha  = 0.011

alpha_shrinkage = ____
print(f"Alpha shrinkage from CAPM → FF3: {alpha_shrinkage:.2%}")

### Q3 — Statistical reality check

The CAPM alpha has t-stat 1.8 (close to "significant"). The FF3 alpha has t-stat 0.5 (essentially zero).

> **📌 Required:**
> ```python
> capm_alpha_significant = ____   # True/False — is |t| > 2?
> ff3_alpha_significant  = ____   # True/False
> ```

(Coerce True to 1.0 and False to 0.0 in the variable assignments so the
submission cell can encode them as floats.)

In [ ]:
capm_t = 1.8
ff3_t  = 0.5

capm_alpha_significant = ____    # 1.0 if |capm_t| > 2 else 0.0
ff3_alpha_significant  = ____
print(f"CAPM alpha 'significant'? {bool(capm_alpha_significant)}")
print(f"FF3 alpha 'significant'?  {bool(ff3_alpha_significant)}")

### Q4 — Memo

Max 5 sentences. Recommend whether your fund should invest. Cite (i) Sharpe with uncertainty, (ii) where the alpha goes when you control for FF3, (iii) the appropriate benchmark for THIS fund.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["fund_sharpe", "fund_sharpe_se", "alpha_shrinkage", "capm_alpha_significant", "ff3_alpha_significant", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "PerformanceEval_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Alpha shrinks as you add factors.** Always benchmark against multi-factor models.
2. **Sharpe estimates have wide CIs.** A 1.0 Sharpe over 10 years has a CI of ~[0.78, 1.22].
3. **Single-factor benchmarks are inadequate** for any active fund with size/value tilts.
4. **Multiple testing is a real concern.** Out of 500 funds, ~25 will show "significant" alpha by chance.
5. **AI runs the regression. You decide what factors to include and how to interpret what's left.**